# Thread in Python
- Thread creation
- Thread Lifecycle
    - NEW
    - RUNNABLE
    - RUNNING
    - TERMINATED
- Custom Thread
- Lock / Synchronization
- Condition Variable (Wait and Notify)
- Producer-Consumer Pattern (Queue)
- Thread Pool
- ThreadLocal Storage
- Semaphore

### Simple Thread Creation (start and join)

In [2]:
# Simple Thread Creation (start and join)
import threading
import time


def task(name):
    for i in range(3):
        print(f"{name} is running {i}\n")
        time.sleep(1)


# Create threads
t1 = threading.Thread(target=task, args=("Thread-1",))
t2 = threading.Thread(target=task, args=("Thread-2",))

# Start threads
t1.start()
t2.start()

# Wait for completion
t1.join()
t2.join()

print("Main thread completed")

Thread-1 is running 0

Thread-2 is running 0

Thread-1 is running 1
Thread-2 is running 1


Thread-2 is running 2
Thread-1 is running 2


Main thread completed


### Custom Thread Creation (Start and Join)

In [4]:
# Custom Thread
import threading
import time


class MyThread(threading.Thread):

    def run(self):
        for i in range(3):
            print(f"{self.name} running {i}\n")
            time.sleep(1)


t1 = MyThread()
t2 = MyThread()

t1.start()
t2.start()

t1.join()
t2.join()

print("Done")

Thread-9 running 0

Thread-10 running 0

Thread-9 running 1
Thread-10 running 1


Thread-10 running 2
Thread-9 running 2


Done


### Lock / Synchronization

#### Example without Lock

In [27]:
import threading

counter = 0


def increment():

    global counter

    for i in range(100000):
        counter += 1


t1 = threading.Thread(target=increment)
t2 = threading.Thread(target=increment)

t1.start()
t2.start()

t1.join()
t2.join()

#Value can be different
print("Value may be different:", counter)

Value may be different: 200000


#### Example with Lock

In [28]:
import threading
import time


counter = 0

# Create lock
lock = threading.Lock()


def increment():

    global counter

    for i in range(100000):

        # Acquire lock (lock.acquire() and lock.release())
        with lock:
            counter += 1


# Create threads
t1 = threading.Thread(target=increment)
t2 = threading.Thread(target=increment)

# Start threads
t1.start()
t2.start()

# Wait for completion
t1.join()
t2.join()

print("Final Counter:", counter)

Final Counter: 200000


#### Another Example

In [29]:
import threading


class BankAccount:

    def __init__(self):
        self.balance = 1000
        self.lock = threading.Lock()

    def withdraw(self, amount):

        with self.lock:

            if self.balance >= amount:
                self.balance -= amount
                print(f"Withdrawn {amount}")

            else:
                print("Insufficient balance")


account = BankAccount()

t1 = threading.Thread(target=account.withdraw, args=(700,))
t2 = threading.Thread(target=account.withdraw, args=(700,))

t1.start()
t2.start()

t1.join()
t2.join()

print("Remaining Balance:", account.balance)

Withdrawn 700
Insufficient balance
Remaining Balance: 300


#### Reentrant Lock (RLock) Example in Python
- A reentrant lock allows the same thread to acquire the same lock multiple times.
- With normal Lock, if a thread already holds the lock and tries to acquire it again, Deadlock occurs

In [30]:
import threading


lock = threading.RLock()


class BankAccount:

    def __init__(self):
        self.balance = 1000

    def withdraw(self, amount):

        with lock:

            print("Withdraw method acquired lock")

            self.update_balance(amount)

    def update_balance(self, amount):

        with lock:

            print("Update balance acquired lock")

            self.balance -= amount

            print("Remaining Balance:", self.balance)


account = BankAccount()

t1 = threading.Thread(target=account.withdraw, args=(200,))

t1.start()
t1.join()

Withdraw method acquired lock
Update balance acquired lock
Remaining Balance: 800


### Condition Variable (Wait and Notify)

In [31]:
import threading
import time


items = []
condition = threading.Condition()


# Producer
def producer():

    for i in range(5):

        time.sleep(1)

        with condition:

            item = f"Item-{i}"

            items.append(item)

            print("Produced:", item)

            # Notify waiting consumer
            condition.notify()


# Consumer
def consumer():

    for i in range(5):

        with condition:

            # Wait until item available
            while len(items) == 0:

                print("Consumer waiting...")

                condition.wait()

            item = items.pop(0)

            print("Consumed:", item)


# Create threads
t1 = threading.Thread(target=producer)
t2 = threading.Thread(target=consumer)

# Start threads
t2.start()
t1.start()

# Wait for completion
t1.join()
t2.join()

print("Completed")

Consumer waiting...
Produced: Item-0
Consumed: Item-0
Consumer waiting...
Produced: Item-1
Consumed: Item-1
Consumer waiting...
Produced: Item-2
Consumed: Item-2
Consumer waiting...
Produced: Item-3
Consumed: Item-3
Consumer waiting...
Produced: Item-4
Consumed: Item-4
Completed


### Producer-Consumer Pattern (Queue)

In [33]:
import threading
import time
from queue import Queue


# Shared queue
q = Queue()


# Producer
def producer():

    for i in range(5):

        item = f"Item-{i}"

        print("Produced:\n", item)

        q.put(item)

        time.sleep(1)


# Consumer
def consumer():

    for i in range(5):

        item = q.get()

        print("Consumed:\n", item)

        q.task_done()

        time.sleep(2)


# Create threads
t1 = threading.Thread(target=producer)
t2 = threading.Thread(target=consumer)

# Start threads
t1.start()
t2.start()

# Wait for completion
t1.join()
t2.join()

print("Completed")

Produced:
 Item-0
Consumed:
 Item-0
Produced:
 Item-1
Consumed:
 Item-1
Produced:
 Item-2
Produced:
 Item-3
Consumed:
 Item-2
Produced:
 Item-4
Consumed:
 Item-3
Consumed:
 Item-4
Completed


### Thread Pool

#### Simple Thread Pool Example

In [34]:
from concurrent.futures import ThreadPoolExecutor
import time


def square(n):
    time.sleep(1)
    return n * n


numbers = [1, 2, 3, 4, 5]

with ThreadPoolExecutor(max_workers=3) as executor:
    results = executor.map(square, numbers)

print(list(results))

[1, 4, 9, 16, 25]


#### Thread Pool With Submit

In [35]:
from concurrent.futures import ThreadPoolExecutor
import time


def task(name):
    time.sleep(2)
    return f"{name} completed"


with ThreadPoolExecutor(max_workers=2) as executor:

    future1 = executor.submit(task, "Job-1")
    future2 = executor.submit(task, "Job-2")

    print(future1.result())
    print(future2.result())

Job-1 completed
Job-2 completed


### ThreadLocal Storage

#### Simple Example

In [38]:
import threading
import time


# Thread local object
local_data = threading.local()


def task(name, value):

    local_data.value = value

    time.sleep(1)

    print(name, "->\n", local_data.value)


t1 = threading.Thread(target=task, args=("Thread-1", 100))
t2 = threading.Thread(target=task, args=("Thread-2", 200))

t1.start()
t2.start()

t1.join()
t2.join()

Thread-1 ->
 100
Thread-2 ->
 200


#### Real Example — User Context

In [40]:
import threading


user_context = threading.local()


def process_request(user):

    user_context.user = user

    print("Processing for:\n", user_context.user)


t1 = threading.Thread(target=process_request, args=("Ambarish",))
t2 = threading.Thread(target=process_request, args=("Rajni",))

t1.start()
t2.start()

Processing for:
 Ambarish
Processing for:
 Rajni


### Semaphore

In [42]:
import threading
import time


# Allow maximum 2 threads at a time
semaphore = threading.Semaphore(2)


def task(name):

    print(f"{name} waiting for access\n")

    with semaphore:

        print(f"{name} acquired semaphore\n")

        time.sleep(3)

        print(f"{name} releasing semaphore\n")


# Create multiple threads
threads = []

for i in range(5):

    t = threading.Thread(target=task, args=(f"Thread-{i}",))

    threads.append(t)

    t.start()


# Wait for completion
for t in threads:
    t.join()


print("Completed")

Thread-0 waiting for access

Thread-0 acquired semaphore

Thread-1 waiting for access

Thread-1 acquired semaphore

Thread-2 waiting for access

Thread-3 waiting for access

Thread-4 waiting for access

Thread-1 releasing semaphore
Thread-0 releasing semaphore

Thread-2 acquired semaphore


Thread-3 acquired semaphore

Thread-2 releasing semaphore
Thread-3 releasing semaphore

Thread-4 acquired semaphore


Thread-4 releasing semaphore

Completed
